# 09. Power BI Data Preparation

## 1. Objective

Prepare validated datasets for an interactive, five-page Power BI report covering sales, product and category performance, regional and customer insights, delivery experience, and payment and commercial analysis.

Reuse the cleaned and analysis-ready datasets while preserving the delivered-order scope and established metric definitions.

## 2. Source Dataset Inspection

### 2.1 Inspect Existing Analysis-Ready Datasets

Inspect the order commercial and payment experience dataset and the delivered order-item dataset to identify their structure and suitability for Power BI modeling.

Then validate:

- Dataset dimensions and available columns.
- The grain of each dataset: what one row represents.
- Missing order IDs and distinct order counts.
- Coverage of delivered orders, including orders without payment records.
- Candidate keys and fields needed for relationships and filtering.

The initial inspection below displays shapes, columns, order-ID counts, and sample rows. Key uniqueness and cross-dataset coverage will be checked in subsequent steps.

In [2]:
# Inspect the prepared order-level and item-level datasets for Power BI modeling.
from pathlib import Path
import pandas as pd

processed_path = Path("../data/processed")

dataset_paths = {
    "Order commercial and payment experience": (processed_path / "payment_and_commercial_analysis" / "order_commercial_payment_experience.parquet"),
    "Delivered order items": (processed_path / "delivered_order_items_analysis_ready.parquet"),
}

for dataset_name, file_path in dataset_paths.items():
    df = pd.read_parquet(file_path)

    print(f"\n{dataset_name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    if "order_id" in df.columns:
        print("Distinct orders:", df["order_id"].nunique())
        print("Missing order IDs:", df["order_id"].isna().sum())

    display(df.head(3))


Order commercial and payment experience
Shape: (96478, 27)
Columns: ['order_id', 'order_merchandise_value', 'order_freight_value', 'expected_order_value', 'order_value_band', 'freight_burden_pct', 'freight_burden_band', 'payment_transaction_count', 'distinct_payment_type_count', 'total_payment_value', 'maximum_payment_installments', 'payment_quality_exception_count', 'flag_order_with_multiple_payments', 'flag_order_with_multiple_payment_types', 'dominant_payment_type', 'dominant_payment_type_value', 'dominant_payment_type_transaction_count', 'dominant_payment_type_maximum_installments', 'payment_coverage', 'average_review_score', 'delivery_duration_days', 'delivery_calendar_difference_days', 'delivery_status', 'flag_review_eligible', 'flag_delivery_duration_eligible', 'flag_delivery_status_eligible', 'experience_coverage']
Distinct orders: 96478
Missing order IDs: 0


,order_id,order_merchandise_value,order_freight_value,expected_order_value,order_value_band,freight_burden_pct,freight_burden_band,payment_transaction_count,distinct_payment_type_count,total_payment_value,...,dominant_payment_type_maximum_installments,payment_coverage,average_review_score,delivery_duration_days,delivery_calendar_difference_days,delivery_status,flag_review_eligible,flag_delivery_duration_eligible,flag_delivery_status_eligible,experience_coverage
0,00010242fe8c5a6d1ba2dd792cb16214,58.9,13.29,72.19,50-100,22.563667,10-25%,1.0,1.0,72.19,...,2.0,both,5.0,7.614421,-9,early,True,True,True,both
1,00018f77f2f0320c557190d7a144bdd3,239.9,19.93,259.83,150-400,8.307628,0-10%,1.0,1.0,259.83,...,3.0,both,4.0,16.216181,-3,early,True,True,True,both
2,000229ec398224ef6ca0657da4fc703e,199.0,17.87,216.87,150-400,8.979899,0-10%,1.0,1.0,216.87,...,5.0,both,5.0,7.948437,-14,early,True,True,True,both



Delivered order items
Shape: (110197, 74)
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'flag_future_shipping_limit_date', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'flag_delivered_missing_customer_date', 'flag_delivered_missing_carrier_date', 'flag_delivered_missing_approval_date', 'flag_delivered_missing_critical_timestamp', 'critical_timestamp_flag_count', 'flag_carrier_before_purchase', 'flag_customer_before_carrier', 'flag_carrier_before_approval', 'flag_lifecycle_timestamp_exception', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'flag_missing_catalog_attribute

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,flag_future_shipping_limit_date,customer_id,order_status,...,delivery_difference_days,delivery_calendar_difference_days,delivery_status,freight_burden_percentage,order_purchase_month,order_purchase_year,order_purchase_quarter,order_purchase_month_number,order_purchase_weekday,order_purchase_weekday_number
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,False,3ce436f183e68e07877b285a838db11a,delivered,...,-8.011250,-9,early,22.563667,2017-09,2017,3,9,Wednesday,2
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,False,f6dd3ec061db4e3987629fe6b26e5cce,delivered,...,-2.330278,-3,early,8.307628,2017-04,2017,2,4,Wednesday,2
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,False,6489ae5e4333f3693df5ad4372dab6d3,delivered,...,-13.444954,-14,early,8.979899,2018-01,2018,1,1,Sunday,6


### 2.2 Validate Dataset Grain and Order Coverage

Validate the keys and order coverage before preparing the Power BI tables.

Then validate:

- The commercial dataset contains one row per order.
- Each order-item combination is unique.
- Neither dataset has missing key values.
- Both datasets contain exactly the same order IDs.
- All item records belong to delivered orders.
- Purchase timestamps and customer attributes are consistent within each order.

In [3]:
# Load the prepared datasets into named DataFrames for Power BI preparation.
order_experience = pd.read_parquet(dataset_paths["Order commercial and payment experience"])

order_items = pd.read_parquet(dataset_paths["Delivered order items"])

# Validate order-level and item-level keys.
print("Duplicate order IDs in order experience:", order_experience["order_id"].duplicated().sum())

print("Duplicate order-item keys:", order_items.duplicated(["order_id", "order_item_id"]).sum())

print("Missing order IDs in order experience:", order_experience["order_id"].isna().sum())

print("Item rows with missing key values:", order_items[["order_id", "order_item_id"]].isna().any(axis=1).sum())

# Compare actual order-ID sets; matching counts alone do not prove coverage.
experience_order_ids = set(order_experience["order_id"].dropna())
item_order_ids = set(order_items["order_id"].dropna())

print("Orders in experience but not items:", len(experience_order_ids - item_order_ids))

print("Orders in items but not experience:", len(item_order_ids - experience_order_ids))

print("All item records are delivered:", order_items["order_status"].eq("delivered").fillna(False).all())

# Check order attributes before extracting one record per order.
order_attribute_columns = [
    "order_purchase_timestamp",
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
]

attribute_counts = (order_items.groupby("order_id")[order_attribute_columns].nunique(dropna=False))

print("\nOrders with conflicting attribute values:")
display(attribute_counts.gt(1).sum().rename("conflicting_order_count"))

print("\nMissing attribute values in item rows:")
display(order_items[order_attribute_columns].isna().sum().rename("missing_row_count"))

Duplicate order IDs in order experience: 0
Duplicate order-item keys: 0
Missing order IDs in order experience: 0
Item rows with missing key values: 0
Orders in experience but not items: 0
Orders in items but not experience: 0
All item records are delivered: True

Orders with conflicting attribute values:


order_purchase_timestamp    0
customer_id                 0
customer_unique_id          0
customer_city               0
customer_state              0
Name: conflicting_order_count, dtype: int64


Missing attribute values in item rows:


order_purchase_timestamp    0
customer_id                 0
customer_unique_id          0
customer_city               0
customer_state              0
Name: missing_row_count, dtype: int64

## 3. Prepare Power BI Model Tables

### 3.1 Prepare the Order-Level Fact Table

Create one row per delivered order by enriching the commercial and
experience dataset with its purchase timestamp and customer attributes.

Retain payment and experience eligibility flags so missing or ineligible
records can be handled consistently in Power BI measures.

Then validate:

- The table contains 96,478 rows.
- Order IDs remain unique and non-null.
- Every order matches its purchase and customer attributes.
- Purchase dates are populated.
- Merchandise and freight totals remain unchanged.

In [4]:
# Extract the validated order attributes from the item-level dataset.
order_attributes = (order_items[
    [
        "order_id",
        "order_purchase_timestamp",
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state",
    ]].drop_duplicates().copy()
)

# Enrich the order-level dataset without changing its grain.
fact_orders = order_experience.merge(order_attributes,on="order_id", how="left", validate="one_to_one",indicator="_attribute_merge",)

# Create a date-only field for the future date-table relationship.
fact_orders["order_purchase_timestamp"] = pd.to_datetime(fact_orders["order_purchase_timestamp"],errors="raise",)

fact_orders["order_purchase_date"] = (fact_orders["order_purchase_timestamp"].dt.normalize())

# Validate row counts, keys, attribute coverage, and commercial totals.
print("fact_orders shape:", fact_orders.shape)
print("Duplicate order IDs:", fact_orders["order_id"].duplicated().sum())
print("Missing order IDs:", fact_orders["order_id"].isna().sum())
print("Orders without matching attributes:", fact_orders["_attribute_merge"].ne("both").sum(),)
print("Missing purchase dates:", fact_orders["order_purchase_date"].isna().sum(),)

for column in ["order_merchandise_value", "order_freight_value"]:
    original_total = order_experience[column].sum()
    prepared_total = fact_orders[column].sum()

    print(f"\n{column} total: R${prepared_total:,.2f}")
    print("Total preserved within R$0.01:", abs(prepared_total - original_total) < 0.01,)

# Remove the temporary merge-validation field.
fact_orders = fact_orders.drop(columns="_attribute_merge")

print("\nFinal fact_orders shape:", fact_orders.shape)
display(fact_orders.head(3))

fact_orders shape: (96478, 34)
Duplicate order IDs: 0
Missing order IDs: 0
Orders without matching attributes: 0
Missing purchase dates: 0

order_merchandise_value total: R$13,221,498.11
Total preserved within R$0.01: True

order_freight_value total: R$2,198,275.64
Total preserved within R$0.01: True

Final fact_orders shape: (96478, 33)


,order_id,order_merchandise_value,order_freight_value,expected_order_value,order_value_band,freight_burden_pct,freight_burden_band,payment_transaction_count,distinct_payment_type_count,total_payment_value,...,flag_review_eligible,flag_delivery_duration_eligible,flag_delivery_status_eligible,experience_coverage,order_purchase_timestamp,customer_id,customer_unique_id,customer_city,customer_state,order_purchase_date
0,00010242fe8c5a6d1ba2dd792cb16214,58.9,13.29,72.19,50-100,22.563667,10-25%,1.0,1.0,72.19,...,True,True,True,both,2017-09-13 08:59:02,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,2017-09-13
1,00018f77f2f0320c557190d7a144bdd3,239.9,19.93,259.83,150-400,8.307628,0-10%,1.0,1.0,259.83,...,True,True,True,both,2017-04-26 10:53:06,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,2017-04-26
2,000229ec398224ef6ca0657da4fc703e,199.0,17.87,216.87,150-400,8.979899,0-10%,1.0,1.0,216.87,...,True,True,True,both,2018-01-14 14:33:31,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,2018-01-14


### 3.2 Prepare the Order-Item Fact Table

Create one row per delivered order item, retaining order, product, and
seller identifiers together with item merchandise and freight values.

Order-level payment, review, and delivery measures remain in fact_orders
to avoid double-counting and item-weighted averages.

Then validate:

- The table contains 110,197 rows.
- Each order_id and order_item_id combination is unique.
- Required identifiers are populated.
- Every item links to an order in fact_orders.
- Item merchandise and freight sums reconcile to each order within R$0.01.

In [5]:
# Select item-level identifiers and commercial values.
fact_order_items = order_items[
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
    ]
].copy()

# Validate the item grain and required identifiers.
print("fact_order_items shape:", fact_order_items.shape)
print("Duplicate order-item keys:", fact_order_items.duplicated(["order_id", "order_item_id"]).sum(),)

print("\nMissing identifiers:")

print(fact_order_items[["order_id", "order_item_id", "product_id", "seller_id"]].isna().sum())

print("\nItems without a matching order:", (~fact_order_items["order_id"].isin(fact_orders["order_id"])).sum(),)

# Aggregate item values to verify commercial totals for every order.
item_totals = (fact_order_items.groupby("order_id", as_index=False).agg(item_merchandise_total=("price", "sum"),
                                                                        item_freight_total=("freight_value", "sum"),))

commercial_check = fact_orders[["order_id", "order_merchandise_value", "order_freight_value"]].merge(item_totals,
                                                                                                     on="order_id",
                                                                                                     how="outer",
                                                                                                     validate="one_to_one", 
                                                                                                     indicator=True,)

print("Orders without matching commercial records:", commercial_check["_merge"].ne("both").sum(),)

for order_column, item_column in [("order_merchandise_value", "item_merchandise_total"), ("order_freight_value", "item_freight_total"),]:
    difference = (commercial_check[order_column] - commercial_check[item_column]).abs()

    matches = (commercial_check[[order_column, item_column]].notna().all(axis=1) & difference.lt(0.01))

    print(f"{order_column}: all orders reconcile:", matches.all())
    print(f"{order_column}: unmatched or outside tolerance:", (~matches).sum())

display(fact_order_items.head(3))

fact_order_items shape: (110197, 6)
Duplicate order-item keys: 0

Missing identifiers:
order_id         0
order_item_id    0
product_id       0
seller_id        0
dtype: int64

Items without a matching order: 0
Orders without matching commercial records: 0
order_merchandise_value: all orders reconcile: True
order_merchandise_value: unmatched or outside tolerance: 0
order_freight_value: all orders reconcile: True
order_freight_value: unmatched or outside tolerance: 0


,order_id,order_item_id,product_id,seller_id,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.0,17.87


### 3.3 Prepare the Product Dimension

Create one row per product sold in the delivered-order scope, retaining
the original category and the prepared analysis category.

Use product_category_name_analysis for report labels so the existing
translation and missing-category handling are preserved.

Then validate:

- Each product has consistent category attributes.
- Product IDs are unique and non-null.
- Analysis category labels are populated.
- Every product in fact_order_items exists in dim_products.

In [6]:
# Check category consistency before creating one row per product.
product_columns = [
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_category_name_analysis",]

product_attribute_counts = (order_items.groupby("product_id")[product_columns[1:]].nunique(dropna=False))

conflicting_products = product_attribute_counts.gt(1).any(axis=1)

print("Products with conflicting categories:", conflicting_products.sum())

assert not conflicting_products.any(), ("Resolve conflicting product categories before creating dim_products.")

# Preserve the category labels established during data preparation.
dim_products = (order_items[product_columns].drop_duplicates().reset_index(drop=True).copy())

# Validate the dimension key and reporting labels.
print("dim_products shape:", dim_products.shape)
print("Duplicate product IDs:", dim_products["product_id"].duplicated().sum())
print("Missing product IDs:", dim_products["product_id"].isna().sum())

invalid_category_labels = (dim_products["product_category_name_analysis"].isna() | dim_products["product_category_name_analysis"].astype("string").str.strip().eq("").fillna(False))

print("Missing or blank analysis categories:", invalid_category_labels.sum())
print("Distinct analysis categories:", dim_products["product_category_name_analysis"].nunique(),)
print("Item rows without a matching product:", (~fact_order_items["product_id"].isin(dim_products["product_id"])).sum(),)

display(dim_products.head(5))

Products with conflicting categories: 0
dim_products shape: (32216, 4)
Duplicate product IDs: 0
Missing product IDs: 0
Missing or blank analysis categories: 0
Distinct analysis categories: 74
Item rows without a matching product: 0


,product_id,product_category_name,product_category_name_english,product_category_name_analysis
0,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,cool_stuff,cool_stuff
1,e5f2d52b802189ee658865ca93d83a8f,pet_shop,pet_shop,pet_shop
2,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,furniture_decor,furniture_decor
3,7634da152a4610f1595efa32f14722fc,perfumaria,perfumery,perfumery
4,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,garden_tools,garden_tools


### 3.4 Prepare the Date Dimension

Create a continuous daily calendar covering the full calendar years
represented by delivered-order purchase dates.

Include year, quarter, month, and weekday attributes for report filtering
and chronological sorting.

Then validate:

- Dates are unique and non-null.
- The calendar contains no gaps.
- Every purchase date in fact_orders exists in dim_date.
- Each year-month label has a consistent numeric sort key.

In [8]:
# Build a continuous calendar covering complete purchase years.
first_purchase_date = fact_orders["order_purchase_date"].min()
last_purchase_date = fact_orders["order_purchase_date"].max()

calendar_start = pd.Timestamp(year=first_purchase_date.year, month=1, day=1)
calendar_end = pd.Timestamp(year=last_purchase_date.year, month=12, day=31)

dim_date = pd.DataFrame({"date": pd.date_range(calendar_start, calendar_end, freq="D")})

# Add calendar attributes and numeric sort keys.
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter_number"] = dim_date["date"].dt.quarter
dim_date["quarter"] = "Q" + dim_date["quarter_number"].astype(str)
dim_date["month_number"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["year_month"] = dim_date["date"].dt.strftime("%b %Y")
dim_date["year_month_sort"] = (dim_date["year"] * 100 + dim_date["month_number"])
dim_date["day_of_month"] = dim_date["date"].dt.day
dim_date["weekday_number"] = dim_date["date"].dt.dayofweek + 1
dim_date["weekday_name"] = dim_date["date"].dt.day_name()

# Validate calendar integrity and purchase-date coverage.
print("dim_date shape:", dim_date.shape)
print("Calendar range:", calendar_start.date(), "to", calendar_end.date())
print("Duplicate dates:", dim_date["date"].duplicated().sum())
print("Missing dates:", dim_date["date"].isna().sum())

print("Calendar has no gaps:", dim_date["date"].diff().dropna().eq(pd.Timedelta(days=1)).all(),)
print("Orders without a matching purchase date:",(~fact_orders["order_purchase_date"].isin(dim_date["date"])).sum(),)
print("Year-month labels have consistent sort keys:",dim_date.groupby("year_month")["year_month_sort"].nunique().eq(1).all(),)

display(dim_date.head(3))
display(dim_date.tail(3))

dim_date shape: (1096, 11)
Calendar range: 2016-01-01 to 2018-12-31
Duplicate dates: 0
Missing dates: 0
Calendar has no gaps: True
Orders without a matching purchase date: 0
Year-month labels have consistent sort keys: True


,date,year,quarter_number,quarter,month_number,month_name,year_month,year_month_sort,day_of_month,weekday_number,weekday_name
0,2016-01-01,2016,1,Q1,1,January,Jan 2016,201601,1,5,Friday
1,2016-01-02,2016,1,Q1,1,January,Jan 2016,201601,2,6,Saturday
2,2016-01-03,2016,1,Q1,1,January,Jan 2016,201601,3,7,Sunday


,date,year,quarter_number,quarter,month_number,month_name,year_month,year_month_sort,day_of_month,weekday_number,weekday_name
1093,2018-12-29,2018,4,Q4,12,December,Dec 2018,201812,29,6,Saturday
1094,2018-12-30,2018,4,Q4,12,December,Dec 2018,201812,30,7,Sunday
1095,2018-12-31,2018,4,Q4,12,December,Dec 2018,201812,31,1,Monday


### 3.5 Prepare the Customer-State Dimension

Create one row per customer state represented in delivered orders.
Use the state recorded on each order for regional analysis.

Then validate:

- State codes are unique.
- State codes are non-null and non-blank.
- Every order has a matching state in the dimension.

In [9]:
# Create a state dimension from delivered-order customer locations.
dim_customer_state = (fact_orders[["customer_state"]].drop_duplicates().sort_values("customer_state").reset_index(drop=True).copy())

# Identify missing or blank state codes without changing source values.
invalid_state_codes = (dim_customer_state["customer_state"].isna() | dim_customer_state["customer_state"].astype("string").str.strip().eq("").fillna(False))

# Validate dimension keys and order coverage.
print("dim_customer_state shape:", dim_customer_state.shape)
print("Duplicate state codes:", dim_customer_state["customer_state"].duplicated().sum(),)
print("Missing or blank state codes:", invalid_state_codes.sum())
print("Orders without a matching customer state:", (~fact_orders["customer_state"].isin(dim_customer_state["customer_state"])).sum(),)

display(dim_customer_state)

dim_customer_state shape: (27, 1)
Duplicate state codes: 0
Missing or blank state codes: 0
Orders without a matching customer state: 0


,customer_state
0,AC
1,AL
2,AM
3,AP
4,BA
5,CE
6,DF
7,ES
8,GO
9,MA


### 3.6 Prepare the Customer Dimension

Create one row per distinct customer in the delivered-order scope,
using customer_unique_id as the customer key.

Keep customer location on fact_orders so regional analysis reflects
the location recorded for each order.

Customer order counts and repeat-purchase measures will be calculated
in Power BI with explicitly defined date-filter behavior.

Then validate:

- Customer identifiers are unique and populated.
- Every order has a matching customer.
- The dimension row count equals the distinct customer count in fact_orders.

In [10]:
# Create one row per customer across delivered orders.
dim_customers = (fact_orders[["customer_unique_id"]].drop_duplicates().sort_values("customer_unique_id").reset_index(drop=True).copy())

# Validate the customer key.
invalid_customer_ids = (dim_customers["customer_unique_id"].isna()| dim_customers["customer_unique_id"].astype("string").str.strip().eq("").fillna(False))

print("dim_customers shape:", dim_customers.shape)
print("Duplicate customer IDs:", dim_customers["customer_unique_id"].duplicated().sum(),)
print("Missing or blank customer IDs:", invalid_customer_ids.sum())
print("Orders without a matching customer:", (~fact_orders["customer_unique_id"].isin(dim_customers["customer_unique_id"])).sum(),)
print("Customer count matches fact_orders:",len(dim_customers) == fact_orders["customer_unique_id"].nunique(),)

display(dim_customers.head(5))

dim_customers shape: (93358, 1)
Duplicate customer IDs: 0
Missing or blank customer IDs: 0
Orders without a matching customer: 0
Customer count matches fact_orders: True


,customer_unique_id
0,0000366f3b9a7992bf8c76cfdf3221e2
1,0000b849f77a49e4a4ce2b2a4ca5be3f
2,0000f46a3911fa3c0805444483337064
3,0000f6ccb0745a6a4b88665a16c9f078
4,0004aac84e0df4da2b147fca70cf8255


### 3.7 Prepare the Seller Dimension

Create one row per seller represented in delivered order items,
retaining the cleaned seller city and state.

Then validate:

- Each seller has consistent location attributes.
- Seller IDs are unique and populated.
- Missing location attributes are reported.
- Every order item has a matching seller.

In [11]:
# Check seller location consistency before creating the dimension.
seller_columns = ["seller_id", "seller_city", "seller_state"]

seller_attribute_counts = (order_items.groupby("seller_id")[["seller_city", "seller_state"]].nunique(dropna=False))

conflicting_sellers = seller_attribute_counts.gt(1).any(axis=1)

print("Sellers with conflicting locations:", conflicting_sellers.sum())

assert not conflicting_sellers.any(), ("Resolve conflicting seller locations before creating dim_sellers.")

# Create one row per seller, preserving cleaned location attributes.
dim_sellers = (order_items[seller_columns].drop_duplicates().sort_values("seller_id").reset_index(drop=True).copy())

# Validate seller identifiers and item coverage.
invalid_seller_ids = (dim_sellers["seller_id"].isna() | dim_sellers["seller_id"].astype("string").str.strip().eq("").fillna(False))

print("dim_sellers shape:", dim_sellers.shape)
print("Duplicate seller IDs:", dim_sellers["seller_id"].duplicated().sum())
print("Missing or blank seller IDs:", invalid_seller_ids.sum())

print("\nMissing or blank location attributes:")
for column in ["seller_city", "seller_state"]:
    invalid_locations = (dim_sellers[column].isna() | dim_sellers[column].astype("string").str.strip().eq("").fillna(False))
    print(f"{column}: {invalid_locations.sum()}")

print("\nItem rows without a matching seller:",(~fact_order_items["seller_id"].isin(dim_sellers["seller_id"])).sum(),)

display(dim_sellers.head(5))

Sellers with conflicting locations: 0
dim_sellers shape: (2970, 3)
Duplicate seller IDs: 0
Missing or blank seller IDs: 0

Missing or blank location attributes:
seller_city: 0
seller_state: 0

Item rows without a matching seller: 0


,seller_id,seller_city,seller_state
0,0015a82c2db000af6aaaf3ae2ecb0532,santo andre,SP
1,001cca7ae9ae17fb1caed9dfb1094831,cariacica,ES
2,002100f778ceb8431b7a1020ff7ab48f,franca,SP
3,003554e2dce176b5555353e4f3555ac8,goiania,GO
4,004c9cd9d87a3c30c522c48c4fc07416,ibitinga,SP


### 3.8 Prepare the Dominant Payment-Method Dimension

Create a payment-method dimension for filtering orders by their
value-dominant payment method.

Preserve the original dominant_payment_type column and create a separate
reporting key. Label orders without payment records as
"Missing payment record"; distinguish these from any other missing assignments.

Then validate:

- Payment coverage and missing dominant-method assignments are inspected.
- Dimension keys are unique and populated.
- Every order has a matching payment-method key.
- The order count remains unchanged.

In [12]:
# Inspect payment coverage before assigning reporting labels.
print("Payment coverage:")
print(fact_orders["payment_coverage"].value_counts(dropna=False))

missing_method = (fact_orders["dominant_payment_type"].isna() | fact_orders["dominant_payment_type"].astype("string").str.strip().eq("").fillna(False))

print("\nOrders with missing dominant payment method:", missing_method.sum())

# Use the previously validated payment record count to identify missing records.
missing_payment_record = (fact_orders["payment_transaction_count"].isna() | fact_orders["payment_transaction_count"].eq(0))

# Preserve the original field and add an explicit reporting key.
fact_orders["dominant_payment_method_key"] = (fact_orders["dominant_payment_type"].astype("string"))

fact_orders.loc[missing_method & missing_payment_record, "dominant_payment_method_key",] = "Missing payment record"

fact_orders.loc[missing_method & ~missing_payment_record,"dominant_payment_method_key",] = "Unassigned payment method"

# Create one row per reporting key.
dim_payment_method = (fact_orders[["dominant_payment_method_key"]].drop_duplicates().sort_values("dominant_payment_method_key").reset_index(drop=True).copy())

# Validate dimension keys and coverage.
print("\ndim_payment_method shape:", dim_payment_method.shape)
print("Duplicate payment-method keys:", dim_payment_method["dominant_payment_method_key"].duplicated().sum(),)

print("Missing payment-method keys:",dim_payment_method["dominant_payment_method_key"].isna().sum(),)
print("Orders without a matching payment method:",(~fact_orders["dominant_payment_method_key"].isin(dim_payment_method["dominant_payment_method_key"])).sum(),)
print("Order count unchanged:", len(fact_orders) == len(order_experience))

print("\nOrders by dominant payment method:")
print(fact_orders["dominant_payment_method_key"].value_counts(dropna=False))

display(dim_payment_method)

Payment coverage:
payment_coverage
both          96477
left_only         1
right_only        0
Name: count, dtype: int64

Orders with missing dominant payment method: 1

dim_payment_method shape: (5, 1)
Duplicate payment-method keys: 0
Missing payment-method keys: 0
Orders without a matching payment method: 0
Order count unchanged: True

Orders by dominant payment method:
dominant_payment_method_key
credit_card               72785
boleto                    19191
voucher                    3017
debit_card                 1484
Missing payment record        1
Name: count, dtype: int64[pyarrow]


,dominant_payment_method_key
0,Missing payment record
1,boleto
2,credit_card
3,debit_card
4,voucher


### 3.9 Inspect Product Category Mapping

Inspect how original product categories map to English and analysis labels.
Identify missing translations and confirm that each original category maps
to a consistent analysis label.

Then validate:

- Distinct category counts are reported for each category field.
- Missing original categories and translations are identified.
- Each original category maps to exactly one analysis label.
- Existing analysis labels are preserved during inspection.

In [13]:
# Compare category counts without changing the prepared labels.
category_columns = [
    "product_category_name",
    "product_category_name_english",
    "product_category_name_analysis",
]

print("Distinct non-null category counts:")
print(dim_products[category_columns].nunique())

# Inspect category mappings and the number of products using each mapping.
category_mapping = (dim_products.groupby(category_columns, dropna=False, observed=True,).size().reset_index(name="product_count").sort_values("product_category_name_analysis"))

# Identify missing or blank original categories and English translations.
missing_original = (dim_products["product_category_name"].isna() | dim_products["product_category_name"].astype("string").str.strip().eq("").fillna(False))

missing_translation = (dim_products["product_category_name_english"].isna() | dim_products["product_category_name_english"].astype("string").str.strip().eq("").fillna(False))

print("\nProducts with missing original category:", missing_original.sum())
print("Products with missing English translation:", missing_translation.sum())

print("\nMappings involving missing categories or translations:")
display(dim_products.loc[missing_original | missing_translation,category_columns,].drop_duplicates())

# Verify consistent mapping from original category to analysis label.
analysis_labels_per_original = (dim_products.groupby("product_category_name", dropna=False, observed=True,)["product_category_name_analysis"].nunique(dropna=False))

print("\nOriginal categories mapped to multiple analysis labels:",analysis_labels_per_original.gt(1).sum(),)

display(category_mapping)

Distinct non-null category counts:
product_category_name             73
product_category_name_english     71
product_category_name_analysis    74
dtype: int64

Products with missing original category: 584
Products with missing English translation: 595

Mappings involving missing categories or translations:


,product_category_name,product_category_name_english,product_category_name_analysis
110,NaN,NaN,unknown
2315,portateis_cozinha_e_preparadores_de_alimentos,NaN,portateis_cozinha_e_preparadores_de_alimentos
7278,pc_gamer,NaN,pc_gamer



Original categories mapped to multiple analysis labels: 0


,product_category_name,product_category_name_english,product_category_name_analysis,product_count
0,agro_industria_e_comercio,agro_industry_and_commerce,agro_industry_and_commerce,71
19,climatizacao,air_conditioning,air_conditioning,121
3,artes,art,art,52
4,artes_e_artesanato,arts_and_craftmanship,arts_and_craftmanship,19
7,audio,audio,audio,58
...,...,...,...,...
69,tablets_impressao_imagem,tablets_printing_image,tablets_printing_image,9
70,telefonia,telephony,telephony,1112
12,brinquedos,toys,toys,1373
73,NaN,NaN,unknown,584


### 3.9.1 Product Category Mapping Findings

The delivered-product dimension contains 74 analysis categories:
71 translated categories, two untranslated categories retained under their
original names, and one unknown category.

The unknown category contains 584 products with missing original categories.
A further 11 products belong to categories without English translations:

- portateis_cozinha_e_preparadores_de_alimentos
- pc_gamer

Each original category maps to exactly one analysis label. All 74 analysis
categories are retained to preserve product coverage.

Power BI category filtering will use product_category_name_analysis.
Category-level reconciliation must use this same field and delivered-order scope.

## 4. Export Power BI Model Tables

### 4.1 Validate and Export Prepared Tables

Export the two fact tables and six dimension tables as CSV files.

Before exporting, verify that table keys are unique and populated, and
that all fact-table relationship keys match their corresponding tables.

Then validate:

- All key and relationship checks pass.
- Each exported file has the expected row and column counts when read back.
- Identifiers are preserved as text.
- Source analysis datasets remain unchanged.

In [14]:
# Register the prepared model tables and their unique keys.
model_tables = {
    "fact_orders": fact_orders,
    "fact_order_items": fact_order_items,
    "dim_products": dim_products,
    "dim_date": dim_date,
    "dim_customer_state": dim_customer_state,
    "dim_customers": dim_customers,
    "dim_sellers": dim_sellers,
    "dim_payment_method": dim_payment_method,
}

table_keys = {
    "fact_orders": ["order_id"],
    "fact_order_items": ["order_id", "order_item_id"],
    "dim_products": ["product_id"],
    "dim_date": ["date"],
    "dim_customer_state": ["customer_state"],
    "dim_customers": ["customer_unique_id"],
    "dim_sellers": ["seller_id"],
    "dim_payment_method": ["dominant_payment_method_key"],
}

# Stop before exporting if a table has invalid or duplicate keys.
for table_name, table in model_tables.items():
    keys = table_keys[table_name]

    assert not table[keys].isna().any().any(), (f"{table_name}: missing key values")
    assert not table[keys].astype("string").apply(lambda column: column.str.strip().eq("")).any().any(), f"{table_name}: blank key values"
    assert not table.duplicated(keys).any(), (f"{table_name}: duplicate keys")

# Validate the keys that will support model relationships.
relationships = [
    ("fact_orders", "order_purchase_date", "dim_date", "date"),
    ("fact_orders", "customer_state", "dim_customer_state", "customer_state"),
    ("fact_orders", "customer_unique_id", "dim_customers", "customer_unique_id"),
    ("fact_orders", "dominant_payment_method_key", "dim_payment_method", "dominant_payment_method_key"),
    ("fact_order_items", "order_id", "fact_orders", "order_id"),
    ("fact_order_items", "product_id", "dim_products", "product_id"),
    ("fact_order_items", "seller_id", "dim_sellers", "seller_id"),
]

for child_name, child_key, parent_name, parent_key in relationships:
    matched = model_tables[child_name][child_key].isin(model_tables[parent_name][parent_key])
    assert matched.all(), (f"Unmatched keys: {child_name}.{child_key} → "f"{parent_name}.{parent_key}")

print("All table-key and relationship checks passed.")

# Export copies, preserving timestamps and using ISO dates for date keys.
export_path = processed_path / "powerbi"
export_path.mkdir(parents=True, exist_ok=True)

export_results = []

for table_name, table in model_tables.items():
    export_table = table.copy()

    if table_name == "fact_orders":
        export_table["order_purchase_date"] = (export_table["order_purchase_date"].dt.strftime("%Y-%m-%d"))
    elif table_name == "dim_date":
        export_table["date"] = (export_table["date"].dt.strftime("%Y-%m-%d"))

    file_path = export_path / f"{table_name}.csv"
    export_table.to_csv(file_path, index=False, encoding="utf-8-sig")

    # Read as text to check file structure and preserve identifier strings.
    read_back = pd.read_csv(file_path, dtype="string", keep_default_na=False)

    assert read_back.shape == export_table.shape, (f"{table_name}: exported shape mismatch")
    assert read_back.columns.tolist() == export_table.columns.tolist(), (f"{table_name}: exported column mismatch")

    for key in table_keys[table_name]:
        expected = export_table[key].astype("string").reset_index(drop=True)
        actual = read_back[key].reset_index(drop=True)

        assert actual.eq(expected).all(), (f"{table_name}: exported key mismatch in {key}")

    export_results.append({
        "table": table_name,
        "rows": len(read_back),
        "columns": len(read_back.columns),
        "structure_and_keys_verified": True,
    })

display(pd.DataFrame(export_results))
print("Export folder:", export_path.resolve())

All table-key and relationship checks passed.


,table,rows,columns,structure_and_keys_verified
0,fact_orders,96478,34,True
1,fact_order_items,110197,6,True
2,dim_products,32216,4,True
3,dim_date,1096,11,True
4,dim_customer_state,27,1,True
5,dim_customers,93358,1,True
6,dim_sellers,2970,3,True
7,dim_payment_method,5,1,True


Export folder: C:\Users\Hp\DataAnalyst\data\processed\powerbi
